# Simple supervised learning benchmark
Train on one league-season, test on many, compare models with accuracy/precision/recall/f1.

In [8]:
import os
import re
import glob
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [11]:
DATA_DIR = os.path.join('data', 'spadl_data')
TARGET_COL = 'scores'  # or 'concedes'
TRAIN_DATASET_KEY = 'serie_a_2015_2016'
TEST_DATASET_KEYS = []  # empty = all except train

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

In [4]:
def list_available_dataset_keys(data_dir=DATA_DIR):
    keys = []
    for fpath in glob.glob(os.path.join(data_dir, 'features_*.h5')):
        key = os.path.basename(fpath)[len('features_'):-len('.h5')]
        if os.path.exists(os.path.join(data_dir, f'labels_{key}.h5')):
            keys.append(key)
    return sorted(set(keys))

def split_league_season(dataset_key):
    m2 = re.match(r'^(.*)_(\d{4}_\d{4})$', dataset_key)
    m1 = re.match(r'^(.*)_(\d{4})$', dataset_key)
    if m2:
        league_slug, season_slug = m2.group(1), m2.group(2)
    elif m1:
        league_slug, season_slug = m1.group(1), m1.group(2)
    else:
        league_slug, season_slug = dataset_key, 'unknown'
    return league_slug.replace('_', ' '), season_slug.replace('_', '/')

def load_xy(dataset_key, target_col=TARGET_COL, data_dir=DATA_DIR):
    df_features = pd.read_hdf(os.path.join(data_dir, f'features_{dataset_key}.h5'), key='features')
    df_labels = pd.read_hdf(os.path.join(data_dir, f'labels_{dataset_key}.h5'), key='labels')
    df = df_features.merge(df_labels, on=['game_id', 'action_id'], how='inner')
    feature_cols = [c for c in df.columns if c not in {'game_id', 'action_id', 'scores', 'concedes'}]
    X = df[feature_cols].replace([float('inf'), float('-inf')], pd.NA).fillna(0)
    y = df[target_col].astype(int)
    return X, y

In [6]:
available_dataset_keys = list_available_dataset_keys(DATA_DIR)
X_train, y_train = load_xy(TRAIN_DATASET_KEY, target_col=TARGET_COL, data_dir=DATA_DIR)
train_feature_cols = list(X_train.columns)
test_dataset_keys = TEST_DATASET_KEYS if TEST_DATASET_KEYS else [k for k in available_dataset_keys if k != TRAIN_DATASET_KEY]
print(test_dataset_keys)

['1_bundesliga_2015_2016', '1_bundesliga_2023_2024', 'african_cup_of_nations_2023', 'champions_league_1970_1971', 'champions_league_1971_1972', 'champions_league_1972_1973', 'champions_league_1999_2000', 'champions_league_2003_2004', 'champions_league_2004_2005', 'champions_league_2006_2007', 'champions_league_2008_2009', 'champions_league_2009_2010', 'champions_league_2010_2011', 'champions_league_2011_2012', 'champions_league_2012_2013', 'champions_league_2013_2014', 'champions_league_2014_2015', 'champions_league_2015_2016', 'champions_league_2016_2017', 'champions_league_2017_2018', 'champions_league_2018_2019', 'copa_america_2024', 'copa_del_rey_1977_1978', 'copa_del_rey_1982_1983', 'copa_del_rey_1983_1984', 'fa_women_s_super_league_2018_2019', 'fa_women_s_super_league_2019_2020', 'fa_women_s_super_league_2020_2021', 'fifa_u20_world_cup_1979', 'fifa_world_cup_1958', 'fifa_world_cup_1962', 'fifa_world_cup_1970', 'fifa_world_cup_1974', 'fifa_world_cup_1986', 'fifa_world_cup_1990', '

In [ ]:
models = {
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Extra Trees': ExtraTreesClassifier(random_state=42, n_jobs=-1),
    'MLP': Pipeline([('scaler', StandardScaler()), ('clf', MLPClassifier(max_iter=300, random_state=42))]),
}

param_grids = {
    'Decision Tree': {
        'max_depth': [8, 12, 20, None],
        'min_samples_leaf': [1, 5, 10],
        'class_weight': [None, 'balanced'],
    },
    'Extra Trees': {
        'n_estimators': [150, 300],
        'max_depth': [None, 20],
        'min_samples_leaf': [1, 5],
        'class_weight': [None, 'balanced_subsample'],
    },
    'MLP': {
        'clf__hidden_layer_sizes': [(64,), (128, 64)],
        'clf__alpha': [0.0001, 0.001],
        'clf__learning_rate_init': [0.001, 0.01],
    },
}

trained_models = {}
tuning_summary = []

for model_name, estimator in models.items():
    search = GridSearchCV(
        estimator=estimator,
        param_grid=param_grids[model_name],
        scoring='f1',
        cv=3,
        n_jobs=-1,
        verbose=1,
    )
    search.fit(X_train, y_train)
    trained_models[model_name] = search.best_estimator_
    tuning_summary.append({
        'model': model_name,
        'best_f1_cv': search.best_score_,
        'best_params': str(search.best_params_),
    })

tuning_table = pd.DataFrame(tuning_summary)
display(tuning_table)
print('Trained tuned models:', list(trained_models.keys()))
print('Train rows:', len(X_train), 'Target:', TARGET_COL)

c:\Users\anma10\PycharmProjects\supervised-learning-statsbomb\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\anma10\PycharmProjects\supervised-learning-statsbomb\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


Trained models: ['Decision Tree', 'Extra Trees', 'Gradient Boosting', 'MLP', 'Lasso']
Train rows: 761736 Target: scores


In [9]:
results_tables_by_model = {}

for model_name, model in trained_models.items():
    rows = []
    for test_key in test_dataset_keys:
        league, season = split_league_season(test_key)
        X_test, y_test = load_xy(test_key, target_col=TARGET_COL, data_dir=DATA_DIR)
        y_pred = model.predict(X_test.reindex(columns=train_feature_cols, fill_value=0))
        rows.append({
            'model': model_name,
            'league': league,
            'season': season,
            'tested_league_year': test_key,
            'accuracy': accuracy_score(y_test, y_pred),
            'precision': precision_score(y_test, y_pred, zero_division=0),
            'recall': recall_score(y_test, y_pred, zero_division=0),
            'f1': f1_score(y_test, y_pred, zero_division=0),
        })
    table = pd.DataFrame(rows).sort_values(['league', 'season']).reset_index(drop=True)
    results_tables_by_model[model_name] = table
    print(f'\n=== {model_name} ===')
    display(table)

comparison_table = pd.concat(results_tables_by_model.values(), ignore_index=True)
comparison_table = comparison_table[['model', 'league', 'season', 'accuracy', 'precision', 'recall', 'f1', 'tested_league_year']]
comparison_table = comparison_table.sort_values(['model', 'league', 'season']).reset_index(drop=True)

print('\n=== Combined table ===')
display(comparison_table)


=== Decision Tree ===


,model,league,season,tested_league_year,accuracy,precision,recall,f1
0,Decision Tree,1 bundesliga,2015/2016,1_bundesliga_2015_2016,0.980883,0.158787,0.145608,0.151912
1,Decision Tree,1 bundesliga,2023/2024,1_bundesliga_2023_2024,0.979380,0.128733,0.133976,0.131303
2,Decision Tree,african cup of nations,2023,african_cup_of_nations_2023,0.981307,0.155054,0.154146,0.154599
3,Decision Tree,champions league,1970/1971,champions_league_1970_1971,0.977876,0.096774,0.150000,0.117647
4,Decision Tree,champions league,1971/1972,champions_league_1971_1972,0.984607,0.125000,0.125000,0.125000
...,...,...,...,...,...,...,...,...
69,Decision Tree,uefa europa league,1988/1989,uefa_europa_league_1988_1989,0.979671,0.129870,0.158730,0.142857
70,Decision Tree,uefa women s euro,2022,uefa_women_s_euro_2022,0.980522,0.164756,0.156889,0.160727
71,Decision Tree,uefa women s euro,2025,uefa_women_s_euro_2025,0.977605,0.201094,0.160656,0.178615
72,Decision Tree,women s world cup,2019,women_s_world_cup_2019,0.981068,0.158460,0.152586,0.155468



=== Extra Trees ===


,model,league,season,tested_league_year,accuracy,precision,recall,f1
0,Extra Trees,1 bundesliga,2015/2016,1_bundesliga_2015_2016,0.988978,0.966030,0.064889,0.121609
1,Extra Trees,1 bundesliga,2023/2024,1_bundesliga_2023_2024,0.988979,1.000000,0.052519,0.099796
2,Extra Trees,african cup of nations,2023,african_cup_of_nations_2023,0.989766,0.975904,0.079024,0.146209
3,Extra Trees,champions league,1970/1971,champions_league_1970_1971,0.990659,1.000000,0.050000,0.095238
4,Extra Trees,champions league,1971/1972,champions_league_1971_1972,0.992303,1.000000,0.125000,0.222222
...,...,...,...,...,...,...,...,...
69,Extra Trees,uefa europa league,1988/1989,uefa_europa_league_1988_1989,0.989836,0.800000,0.063492,0.117647
70,Extra Trees,uefa women s euro,2022,uefa_women_s_euro_2022,0.988810,0.921569,0.064120,0.119898
71,Extra Trees,uefa women s euro,2025,uefa_women_s_euro_2025,0.985655,0.980392,0.054645,0.103520
72,Extra Trees,women s world cup,2019,women_s_world_cup_2019,0.989210,0.921053,0.060345,0.113269



=== Gradient Boosting ===


...

,model,league,season,tested_league_year,accuracy,precision,recall,f1
0,MLP,1 bundesliga,2015/2016,1_bundesliga_2015_2016,0.989482,0.841328,0.130063,0.225296
1,MLP,1 bundesliga,2023/2024,1_bundesliga_2023_2024,0.989416,0.795775,0.121115,0.210233
2,MLP,african cup of nations,2023,african_cup_of_nations_2023,0.990167,0.829545,0.142439,0.243131
3,MLP,champions league,1970/1971,champions_league_1970_1971,0.991150,1.000000,0.100000,0.181818
4,MLP,champions league,1971/1972,champions_league_1971_1972,0.992303,1.000000,0.125000,0.222222
...,...,...,...,...,...,...,...,...
69,MLP,uefa europa league,1988/1989,uefa_europa_league_1988_1989,0.990344,0.714286,0.158730,0.259740
70,MLP,uefa women s euro,2022,uefa_women_s_euro_2022,0.989199,0.759690,0.133697,0.227378
71,MLP,uefa women s euro,2025,uefa_women_s_euro_2025,0.986202,0.750000,0.134426,0.227989
72,MLP,women s world cup,2019,women_s_world_cup_2019,0.989535,0.751295,0.125000,0.214339



=== Lasso ===


,model,league,season,tested_league_year,accuracy,precision,recall,f1
0,Lasso,1 bundesliga,2015/2016,1_bundesliga_2015_2016,0.712555,0.029909,0.745864,0.057512
1,Lasso,1 bundesliga,2023/2024,1_bundesliga_2023_2024,0.678128,0.026523,0.747053,0.051227
2,Lasso,african cup of nations,2023,african_cup_of_nations_2023,0.719810,0.028399,0.730732,0.054674
3,Lasso,champions league,1970/1971,champions_league_1970_1971,0.695182,0.019231,0.600000,0.037267
4,Lasso,champions league,1971/1972,champions_league_1971_1972,0.680594,0.023609,0.875000,0.045977
...,...,...,...,...,...,...,...,...
69,Lasso,uefa europa league,1988/1989,uefa_europa_league_1988_1989,0.714552,0.028488,0.777778,0.054964
70,Lasso,uefa women s euro,2022,uefa_women_s_euro_2022,0.711985,0.031996,0.793997,0.061512
71,Lasso,uefa women s euro,2025,uefa_women_s_euro_2025,0.689349,0.035757,0.750820,0.068263
72,Lasso,women s world cup,2019,women_s_world_cup_2019,0.700231,0.028433,0.761207,0.054819



=== Combined table ===


,model,league,season,accuracy,precision,recall,f1,tested_league_year
0,Decision Tree,1 bundesliga,2015/2016,0.980883,0.158787,0.145608,0.151912,1_bundesliga_2015_2016
1,Decision Tree,1 bundesliga,2023/2024,0.979380,0.128733,0.133976,0.131303,1_bundesliga_2023_2024
2,Decision Tree,african cup of nations,2023,0.981307,0.155054,0.154146,0.154599,african_cup_of_nations_2023
3,Decision Tree,champions league,1970/1971,0.977876,0.096774,0.150000,0.117647,champions_league_1970_1971
4,Decision Tree,champions league,1971/1972,0.984607,0.125000,0.125000,0.125000,champions_league_1971_1972
...,...,...,...,...,...,...,...,...
365,MLP,uefa europa league,1988/1989,0.990344,0.714286,0.158730,0.259740,uefa_europa_league_1988_1989
366,MLP,uefa women s euro,2022,0.989199,0.759690,0.133697,0.227378,uefa_women_s_euro_2022
367,MLP,uefa women s euro,2025,0.986202,0.750000,0.134426,0.227989,uefa_women_s_euro_2025
368,MLP,women s world cup,2019,0.989535,0.751295,0.125000,0.214339,women_s_world_cup_2019


In [10]:
for model_name, table in results_tables_by_model.items():
    model_slug = model_name.lower().replace(' ', '_')
    table.to_csv(os.path.join(DATA_DIR, f'eval_{TARGET_COL}_train_{TRAIN_DATASET_KEY}_{model_slug}_simple.csv'), index=False)

comparison_table.to_csv(os.path.join(DATA_DIR, f'eval_{TARGET_COL}_train_{TRAIN_DATASET_KEY}_all_models_simple.csv'), index=False)
print('Saved simple CSV outputs.')

Saved simple CSV outputs.
